# Work with time series · Student

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 21. Work with time series

Distinguish calendar dates, observed records, windows, and changes over time.

Before you start: Parse dates and understand missing values and sorting.

Official reference: https://pandas.pydata.org/docs/user_guide/timeseries.html

### M21C01. Expose a missing day

A DatetimeIndex lets pandas work with calendar dates. asfreq("D") requests a daily index between the first and last observations. The missing October 4 row appears as unknown, not zero.

**Read:** What does freq="D" represent?

Your explanation: _write here_.

**Predict:** How many rows will result contain?

Your explanation: _write here_.

**Explain:** Why not fill the missing day with zero automatically?

Your explanation: _write here_.



In [ ]:
# Predict before running: How many rows will result contain?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
result = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
print(result)

**Your turn:** Build the daily Series and select only entries whose visitor count is missing. Store that filtered Series, retaining its date index, in result and print it.

For the check, keep these result names: `result`.

**Hint:** Filter the daily Series with its isna() mask.

**Watch for this:** asfreq does not aggregate. Ensure the date index is unique before requesting a frequency.

In [ ]:
# Change this code to complete the task.
# Question: Build the daily Series and select only entries whose visitor count is missing. Store that filtered Series, retaining its date index, in result and print it.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
result = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
print(result)

**Quick check:** What can asfreq("D") reveal?

1. Missing calendar dates
2. The true counts for missing days
3. Why the sensor failed

**Explain what you learned:** A gap in records is not automatically a zero.

### M21C02. Resample into calendar bins

resample groups observations into time bins. Here each bin starts at a three-day boundary measured from the first day. Report the number of observations beside the total so a partial bin is visible.

**Read:** What is grouped by resample("3D")?

Your explanation: _write here_.

**Predict:** What total is recorded for October 1 through October 3?

Your explanation: _write here_.

**Explain:** Is the total of 58 for the next bin a complete three-day total?

Your explanation: _write here_.



In [ ]:
# Predict before running: What total is recorded for October 1 through October 3?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
s = df.set_index("Date")["Visitors"].sort_index()
groups = s.resample("3D")
result = pd.DataFrame({"Recorded_Total": groups.sum(min_count=1), "Observed_Days": groups.count()})
print(result)

**Your turn:** Resample to individual days and use sum(min_count=1) so October 4 stays missing. Store only the daily total Series, indexed by date, in result.

For the check, keep these result names: `result`.

**Hint:** Change "3D" to "D" and require at least one known value for a sum.

**Watch for this:** Time-bin labels and boundaries depend on frequency and options; state the reporting period.

In [ ]:
# Change this code to complete the task.
# Question: Resample to individual days and use sum(min_count=1) so October 4 stays missing. Store only the daily total Series, indexed by date, in result.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
s = df.set_index("Date")["Visitors"].sort_index()
groups = s.resample("3D")
result = pd.DataFrame({"Recorded_Total": groups.sum(min_count=1), "Observed_Days": groups.count()})
print(result)

**Quick check:** Why show Observed_Days beside totals?

1. To reveal incomplete bins
2. To make every bin complete
3. To replace missing days

**Explain what you learned:** Time summaries need a coverage check.

### M21C03. Calculate a rolling mean

A rolling window calculates a statistic repeatedly as it moves through the data. On this daily grid, rolling(3) means three calendar rows. min_periods=3 requires three known counts, so windows crossing the missing day stay unknown.

**Read:** What does min_periods=3 require?

Your explanation: _write here_.

**Predict:** What is the rolling mean on October 7?

Your explanation: _write here_.

**Explain:** Would rolling(3) mean three days on the original irregular index?

Your explanation: _write here_.



In [ ]:
# Predict before running: What is the rolling mean on October 7?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = pd.DataFrame({"Visitors": daily, "Mean_3_Days": daily.rolling(3, min_periods=3).mean()})
print(result.round(2))

**Your turn:** Calculate a two-day rolling mean requiring two known daily counts. Store only the resulting mean Series, indexed by date, in result.

For the check, keep these result names: `result`.

**Hint:** Apply rolling(2, min_periods=2).mean() to the daily Series and save that Series directly.

**Watch for this:** Sort time data before rolling. Otherwise the window follows an unintended order.

In [ ]:
# Change this code to complete the task.
# Question: Calculate a two-day rolling mean requiring two known daily counts. Store only the resulting mean Series, indexed by date, in result.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = pd.DataFrame({"Visitors": daily, "Mean_3_Days": daily.rolling(3, min_periods=3).mean()})
print(result.round(2))

**Quick check:** Without a regular daily grid, rolling(3) means what?

1. Exactly three calendar days
2. Three rows
3. Three months

**Explain what you learned:** Window size and missing-data rules both affect a moving average.

### M21C04. Compare with a previous period

shift(1) places the previous row value beside the current row. pct_change(fill_method=None) calculates fractional change without filling gaps. Multiplying by 100 expresses that change as a percentage.

**Read:** Why is the first change missing?

Your explanation: _write here_.

**Predict:** What percentage change is shown on October 2?

Your explanation: _write here_.

**Explain:** Why is the October 5 change unknown?

Your explanation: _write here_.



In [ ]:
# Predict before running: What percentage change is shown on October 2?
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = pd.DataFrame({"Visitors": daily, "Previous_Day": daily.shift(1), "Change_Percent": daily.pct_change(fill_method=None) * 100})
print(result.round(2))

**Your turn:** Calculate the signed difference in visitor counts from two calendar days earlier using shift(2). Store the difference Series, indexed by date, in result.

For the check, keep these result names: `result`.

**Hint:** Subtract the shifted daily Series from the original daily Series.

**Watch for this:** Percent change from a previous zero is undefined; inspect zero denominators.

In [ ]:
# Change this code to complete the task.
# Question: Calculate the signed difference in visitor counts from two calendar days earlier using shift(2). Store the difference Series, indexed by date, in result.
df = attendance()
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
daily = df.set_index("Date")["Visitors"].sort_index().asfreq("D")
result = pd.DataFrame({"Visitors": daily, "Previous_Day": daily.shift(1), "Change_Percent": daily.pct_change(fill_method=None) * 100})
print(result.round(2))

**Quick check:** Why multiply pct_change by 100?

1. It returns fractional change
2. It returns rounded percentage points
3. It returns a count

**Explain what you learned:** Be explicit about the time interval and denominator.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates